In [ ]:
!pip install transformers datasets accelerate pandas sympy

# Step 1: Load and prepare emotion dataset

In [ ]:
import pandas as pd
from datasets import Dataset

# Load dataset
df = pd.read_csv("emotion_predictions_full.csv")

# Map labels to emotions
label_map = {
    0: "sadness",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise"
}

df["emotion"] = df["predicted_label"].map(label_map).fillna("neutral")

# =========================
#  IMPROVED OUTPUT FUNCTION
# =========================
def create_output(row):
    text = row["text"]
    emotion = row["emotion"]

    return (
        f"Reason: The user is feeling {emotion} because \"{text}\". "
        f"Feedback: Provide a helpful and supportive suggestion."
    )

# =========================
#  BETTER INSTRUCTION
# =========================
df["instruction"] = "Explain the emotion using the text context and provide helpful feedback"

# =========================
#  INPUT (TEXT + EMOTION)
# =========================
df["input"] = (
    "Text: " + df["text"].astype(str) +
    " | Emotion: " + df["emotion"].astype(str)
)

# =========================
#  OUTPUT (NOW CONTEXT-AWARE)
# =========================
df["output"] = df.apply(create_output, axis=1)

# =========================
#  CREATE DATASET
# =========================
dataset = Dataset.from_pandas(df[["instruction", "input", "output"]])

dataset_split = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Preview
df[["instruction", "input", "output"]].sample(5, random_state=42)

# Step 2: Load pretrained GPT-2 tokenizer and model

In [ ]:
!pip install sympy==1.12
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

model_name = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name)
model.config.pad_token_id = tokenizer.pad_token_id

# Step 3: Tokenize dataset

In [ ]:
BEHAVIOR_PROMPT = (
    "You are an emotion reasoning assistant. Always identify emotional signals from the input text, "
    "explain the likely reason in 1-2 sentences, and provide practical, respectful improvement feedback. "
    "Do not be rude, judgmental, or overly verbose. Keep responses clear and supportive."
)

def build_prompt(instruction, input_text, output_text):
    return (
        f"### System Behavior:\n{BEHAVIOR_PROMPT}\n\n"
        + f"### Instruction:\n{instruction}\n\n"
        + f"### Input:\n{input_text}\n\n"
        + f"### Response:\n{output_text}"
    )

def tokenize_function(examples):
    prompts = [
        f"### System Behavior:\n{BEHAVIOR_PROMPT}\n\n"
        f"### Instruction:\n{inst}\n\n"
        f"### Input:\n{inp}\n\n"
        f"### Response:\n"
        for inst, inp in zip(examples["instruction"], examples["input"])
    ]

    full_texts = [
        p + out for p, out in zip(prompts, examples["output"])
    ]

    tokenized = tokenizer(
        full_texts,
        padding="max_length",
        truncation=True,
        max_length=128
    )

    # No masking: train on the full token sequence.
    tokenized["labels"] = [ids.copy() for ids in tokenized["input_ids"]]
    return tokenized

tokenized_train = train_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "input", "output"])
tokenized_eval = eval_dataset.map(tokenize_function, batched=True, remove_columns=["instruction", "input", "output"])

What MASKING DOES
labels[i][:len(prompt_ids)] = [-100] * len(prompt_ids)

👉 This tells PyTorch:

“IGNORE loss for input tokens”

🧠 How training changes
❌ Without Masking

Model sees:

### Input: Text: I am sad | Emotion: sadness
### Response: Reason: ...

Learns:

predict → "Text: I am sad..."

👉 BAD ❌

✅ With Masking

Model sees:

INPUT (ignored)
OUTPUT (learn this)

Learns:

predict → "Reason: The user is feeling sadness..."

👉 GOOD ✅

❌ Without masking:

Teacher says:

“Repeat the whole question + answer”

Student learns:

“Just copy everything”

✅ With masking:

Teacher says:

“Ignore question, only learn answer”

Student learns:

“How to answer properly”

# Step 4: Create data collator

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,  # LoRA attention dimension
    lora_alpha=16,  # Alpha parameter for LoRA scaling
    target_modules=["c_attn", "c_proj"],  # Modules to apply LoRA to
    lora_dropout=0.05,  # Dropout probability for LoRA layers
    bias="none",  # Bias type for LoRA layers
    task_type="CAUSAL_LM"  # Task type for causal language modeling
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# Step 5 and 6: Training arguments and Trainer

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./fine-tuned-gpt2",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    save_steps=1000,
    save_total_limit=2,
    logging_steps=100,
    eval_strategy="epoch",
    eval_steps=100,
    learning_rate=5e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator
)

# Step 7 and 8: Train and save model

In [ ]:
trainer.train()

In [ ]:
model.save_pretrained("./fine-tuned-gpt2")
tokenizer.save_pretrained("./fine-tuned-gpt2")

# Step 9: Inference with fine-tuned model

In [ ]:
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel
from peft import PeftModel

# =========================
#  LOAD MODEL CORRECTLY
# =========================

base_model = GPT2LMHeadModel.from_pretrained("gpt2")

model = PeftModel.from_pretrained(
    base_model,
    "./fine-tuned-gpt2/"   # your checkpoint path
)

tokenizer = GPT2Tokenizer.from_pretrained("./fine-tuned-gpt2")

# Set pad token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

model.eval()

# =========================
#  PROMPT (same as training)
# =========================

BEHAVIOR_PROMPT = (
    "You are an emotion reasoning assistant. Always identify emotional signals from the input text, "
    "explain the likely reason in 1-2 sentences, and provide practical, respectful improvement feedback. "
    "Do not be rude, judgmental, or overly verbose. Keep responses clear and supportive."
)

# =========================
#  GENERATION FUNCTION
# =========================

def generate_response(text, emotion):
    prompt = (
        f"### System Behavior:\n{BEHAVIOR_PROMPT}\n\n"
        f"### Instruction:\nExplain the emotion and give improvement feedback\n\n"
        f"### Input:\nText: {text} | Emotion: {emotion}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=80,
            temperature=0.6,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # =========================
    #  CLEAN OUTPUT
    # =========================
    if "### Response:" in generated_text:
        response = generated_text.split("### Response:")[-1]
    else:
        response = generated_text

    # Remove unwanted repeating patterns
    response = response.split("###")[0]

    return response.strip()

# =========================
#  TEST WITH CUSTOM INPUT
# =========================

text = "I failed my exam and feel really bad"
emotion = "sadness"

output = generate_response(text, emotion)

print("\n=== MODEL OUTPUT ===")
print(output)


# =========================
#  MORE TEST CASES
# =========================

print("\n--- More Tests ---")

print(generate_response("I am so happy today!", "joy"))
print(generate_response("This product is terrible and frustrating", "anger"))
print(generate_response("I feel nervous about my interview", "fear"))
print(generate_response("Wow I didn't expect that!", "surprise"))

In [ ]:
generate_response("I am so happy today!", "joy")
generate_response("This product is terrible and frustrating", "anger")
generate_response("I feel nervous about my interview", "fear")
generate_response("Wow I didn't expect that!", "surprise")